In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')  # Columns: e.g., 'filename', 'label'
test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')    # Columns: e.g., 'filename', 'label'

classes = train_df['Label'].unique()
train_subset_df = pd.DataFrame()      

for label in classes:
    class_df = train_df[train_df['Label'] == label]
    sampled = class_df.sample(n=20, random_state=42) 
    train_subset_df = pd.concat([train_subset_df, sampled])

image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'  
train_subset_df['image_path'] = image_dir + train_subset_df['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

print(f"Training subset shape: {train_subset_df.shape}")
print(f"Test set shape: {test_df.shape}")


In [ ]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.get_device_name(0))  # Prints GPU name, e.g., "NVIDIA GeForce RTX 3080"


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class SimpleImageDataset(Dataset):
    def __init__(self, df, image_processor):
        self.image_paths = df['image_path'].tolist()
        # Ensure your DataFrame's label column is also named 'Label'
        self.labels = df['Label'].tolist()
        self.processor = image_processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Return the raw PIL image and label
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        return {"image": image, "label": label}


In [ ]:
def create_collate_fn(processor):
    def collate_fn(batch):
        # 'batch' is a list of dictionaries: [{'image': img1, 'label': 1}, {'image': img2, 'label': 5}, ...]
        
        # Extract the images and labels from the list of dictionaries
        images = [item['image'] for item in batch]
        labels = [item['label'] for item in batch]
        
        # Use the processor to create the 4D tensor batch. It handles everything.
        processed_batch = processor(images=images, return_tensors="pt")
        
        # Add the labels to the batch
        processed_batch['label'] = torch.tensor(labels)
        
        return processed_batch
    return collate_fn

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor

# --- SETUP (Do this once) ---
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

# 1. Create the datasets
# Use your actual dataframes here
train_ds_manual = SimpleImageDataset(train_subset_df, processor)
test_ds_manual = SimpleImageDataset(test_df, processor)

# 2. Create the collate function
my_collate_fn = create_collate_fn(processor)

# 3. Create the DataLoaders with the collate function
# This DataLoader now produces perfectly formed batches
train_dataloader_manual = DataLoader(train_ds_manual, batch_size=8, collate_fn=my_collate_fn)
test_dataloader_manual = DataLoader(test_ds_manual, batch_size=8, collate_fn=my_collate_fn)


# --- REVISED get_embeddings FUNCTION ---
def get_embeddings(model, dataloader, has_labels=True):
    model.eval()
    embeddings = []
    labels_list = [] if has_labels else None

    with torch.no_grad():
        for batch in dataloader:
            # The batch is already a perfect dictionary of tensors!
            # Just move the data to the GPU.
            inputs = {
                'pixel_values': batch['pixel_values'].to('cuda')
            }
            
            outputs = model(**inputs)
            emb = outputs.logits
            embeddings.append(emb.cpu().numpy())
            
            if has_labels:
                labels_list.append(batch['label'].numpy())

    embeddings = np.vstack(embeddings)
    if has_labels:
        labels_list = np.hstack(labels_list)
        return embeddings, labels_list
    
    return embeddings

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification, ViTConfig, Trainer, TrainingArguments
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()

In [ ]:
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)
test_emb_pre, test_labels_pre = get_embeddings(model, test_dataloader_manual)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# Original tuned kernel (may still hit upper bound)
tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))

# Optional: Tighter bounds to force a more reasonable length_scale
# tight_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))

gp_pre = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_pre.fit(train_emb_pre_scaled, train_labels_pre)
pred_gp_pre = gp_pre.predict(test_emb_pre_scaled)
acc_gp_pre = accuracy_score(test_labels_pre, pred_gp_pre)
print(f"GP on Scaled Pretrained ViT Embeddings - Test Accuracy: {acc_gp_pre:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Scale the data (still good practice)
scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# 2. Train a simple, powerful linear model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(train_emb_pre_scaled, train_labels_pre)

# 3. Evaluate
accuracy_log_reg_pretrained = log_reg.score(test_emb_pre_scaled, test_labels_pre)
print(f"Logistic Regression on Pretrained Embeddings - Test Accuracy: {accuracy_log_reg_pretrained:.4f}")

In [ ]:
from sklearn.gaussian_process.kernels import DotProduct

linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

gp_linear = GaussianProcessClassifier(kernel=linear_kernel, random_state=42, n_jobs=-1)
gp_linear.fit(train_emb_pre_scaled, train_labels_pre)

acc_gp_linear = gp_linear.score(test_emb_pre_scaled, test_labels_pre)
print(f"\nGP with Linear Kernel on Pretrained Embeddings - Test Accuracy: {acc_gp_linear:.4f}")

In [ ]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

pca = PCA(n_components=64, random_state=42)  # Or try 50, 100
train_emb_pca = pca.fit_transform(train_emb_pre_scaled)
test_emb_pca = pca.transform(test_emb_pre_scaled)

tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))  # Tighter bounds
gp_pre = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_pre.fit(train_emb_pca, train_labels_pre)
pred_gp_pre = gp_pre.predict(test_emb_pca)
acc_gp_pre = accuracy_score(test_labels_pre, pred_gp_pre)
print(f"GP on PCA-Reduced Scaled Pretrained Embeddings - Test Accuracy: {acc_gp_pre:.4f}")

In [ ]:
class ViTWithGPHead:
    def __init__(self, vit_model_path, kernel, num_labels, use_pca=False, pca_components=64):
        # Load the ViT model (from saved or pretrained)
        self.vit = ViTForImageClassification.from_pretrained(vit_model_path, num_labels=num_labels, ignore_mismatched_sizes=True).to('cuda')
        self.vit.classifier = nn.Identity()  # Bypass classifier to get 768-dim embeddings
        # Freeze the backbone
        for param in self.vit.parameters():
            param.requires_grad = False
        self.gp = GaussianProcessClassifier(kernel=kernel, random_state=42, n_jobs=-1)
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_components, random_state=42) if use_pca else None
        self.use_pca = use_pca

    def fit(self, train_dataloader):
        # Extract train embeddings
        self.vit.eval()
        embeddings = []
        labels_list = []
        with torch.no_grad():
            for batch in train_dataloader:
                inputs = {'pixel_values': batch['pixel_values'].to('cuda')}
                outputs = self.vit(**inputs)
                emb = outputs.logits.cpu().numpy()  # 768-dim embeddings
                embeddings.append(emb)
                labels_list.append(batch['label'].numpy())
        train_emb = np.vstack(embeddings)
        train_labels = np.hstack(labels_list)
        
        # Scale
        train_emb_scaled = self.scaler.fit_transform(train_emb)
        
        # Optional PCA (for RBF kernel in high dims)
        if self.use_pca:
            train_emb_scaled = self.pca.fit_transform(train_emb_scaled)
        
        # Fit GP head
        self.gp.fit(train_emb_scaled, train_labels)
        print("GP head fitted successfully.")

    def predict(self, test_dataloader):
        # Extract test embeddings
        self.vit.eval()
        embeddings = []
        labels_list = []
        with torch.no_grad():
            for batch in test_dataloader:
                inputs = {'pixel_values': batch['pixel_values'].to('cuda')}
                outputs = self.vit(**inputs)
                emb = outputs.logits.cpu().numpy()
                embeddings.append(emb)
                labels_list.append(batch['label'].numpy())
        test_emb = np.vstack(embeddings)
        test_labels = np.hstack(labels_list)
        
        # Scale and optional PCA
        test_emb_scaled = self.scaler.transform(test_emb)
        if self.use_pca:
            test_emb_scaled = self.pca.transform(test_emb_scaled)
        
        # Predict with GP
        preds = self.gp.predict(test_emb_scaled)
        probs = self.gp.predict_proba(test_emb_scaled)  # Optional: Get probabilities/uncertainties
        return preds, test_labels, probs

In [ ]:
!pip install gpytorch

In [ ]:
# RBF kernel (use PCA to avoid underfitting in 768 dims)
tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))  # Tighter bounds
vit_gp_pre_rbf = ViTWithGPHead('/kaggle/input/models/pretrained_vit/pretrained_vit', tuned_kernel, len(classes), use_pca=True, pca_components=5)
vit_gp_pre_rbf.fit(train_dataloader_manual)
preds_rbf, test_labels_rbf, probs_rbf = vit_gp_pre_rbf.predict(test_dataloader_manual)
acc_rbf = accuracy_score(test_labels_rbf, preds_rbf)
print(f"GP (RBF) Head on Frozen Pretrained ViT - Test Accuracy (PCA with 5 components): {acc_rbf:.4f}")

linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

for i in [5, 10, 64, 100, 200]:
    vit_gp_pre_linear = ViTWithGPHead('/kaggle/input/models/pretrained_vit/pretrained_vit', linear_kernel, len(classes), use_pca = True, pca_components=64)
    vit_gp_pre_linear.fit(train_dataloader_manual)
    preds_linear, test_labels_linear, probs_linear = vit_gp_pre_linear.predict(test_dataloader_manual)
    acc_linear = accuracy_score(test_labels_linear, preds_linear)
    print(f"GP (Linear) Head on Frozen Pretrained ViT, PCA with {i} components - Test Accuracy: {acc_linear:.4f}")

In [ ]:
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, MultitaskVariationalStrategy, VariationalStrategy
from gpytorch.mlls import VariationalELBO
import torchimport torch.nn as nn

class MultitaskGPHead(ApproximateGP):
    def __init__(self, num_inducing, feature_dim, num_classes, kernel='rbf'):
        # Use double precision for numerical stability
        inducing_points = torch.randn(num_inducing, feature_dim, dtype=torch.float64).to('cuda')
        
        # Use CholeskyVariationalDistribution instead of Natural for better stability
        base_variational_distribution = CholeskyVariationalDistribution(num_inducing)
        
        base_variational_strategy = VariationalStrategy(
            self, inducing_points, base_variational_distribution, learn_inducing_locations=True
        )
        variational_strategy = MultitaskVariationalStrategy(base_variational_strategy, num_tasks=num_classes)
        super().__init__(variational_strategy)
        
        batch_shape = torch.Size([num_classes])
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        
        # Corrected kernel initialization
        if kernel == 'rbf':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RBFKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'linear':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.LinearKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'cosine':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.CosineKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )

        elif kernel == 'matern':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.MaternKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'periodic':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PeriodicKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'piecewise_poly':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PiecewisePolynomialKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'polynomial':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PolynomialKernel(power=2, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'RQ':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RQKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'spectral_mixture':
            self.covar_module = gpytorch.kernels.SpectralMixtureKernel(
                num_mixtures=4, ard_num_dims=feature_dim, batch_shape=batch_shape
            )
        else:
            raise ValueError(f"Unsupported kernel: {kernel}")

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Attach to frozen ViT (keep your code)
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()
for param in model.parameters():
    param.requires_grad = False

# Extract train embeddings once
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)

# Convert to double precision for numerical stability
train_emb_tensor = torch.tensor(train_emb_pre, dtype=torch.float64).to('cuda')
train_labels_tensor = torch.tensor(train_labels_pre, dtype=torch.long).to('cuda')

# Updated kernel list (removed 'constant' as it causes singularity issues)
kernels = [
    ['linear', 0.08],
    ['polynomial', 0.3],
    ['rbf', 0.01],
           
           ['piecewise_poly', 0.0001],
           ['matern', 0.001], 
           ['RQ', 0.001],
           ['cosine', 0.00001],
           ['spectral_mixture', 0.0001],
           ['periodic', 0.0001],]

# Calculate optimal number of inducing points (25% of data as heuristic)
num_inducing = max(100, int(0.25 * len(train_emb_tensor)))  # At least 100, at most 25% of data
print(f"Using {num_inducing} inducing points out of {len(train_emb_tensor)} training points")

# Increase jitter for numerical stability
with gpytorch.settings.cholesky_jitter(1e-4):  # Increased from default 1e-6
    for kernel in kernels:
        print(f"--------------------------- This is the {kernel} kernel ----------------------------")
        
        # Init GP head with reduced inducing points
        gp_head = MultitaskGPHead(
            num_inducing=num_inducing, 
            feature_dim=768, 
            num_classes=len(classes), 
            kernel=kernel[0]
        ).to('cuda').double()  # Use double precision
        
        # Train GP
        likelihood = gpytorch.likelihoods.SoftmaxLikelihood(
            num_features=len(classes), 
            num_classes=len(classes)
        ).to('cuda').double()  # Use double precision
        
        import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, MultitaskVariationalStrategy, VariationalStrategy
from gpytorch.mlls import VariationalELBO
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score

class MultitaskGPHead(ApproximateGP):
    def __init__(self, num_inducing, feature_dim, num_classes, kernel='rbf'):
        inducing_points = torch.randn(num_inducing, feature_dim, dtype=torch.float64).to('cuda')
        base_variational_distribution = CholeskyVariationalDistribution(num_inducing)
        base_variational_strategy = VariationalStrategy(
            self, inducing_points, base_variational_distribution, learn_inducing_locations=True
        )
        variational_strategy = MultitaskVariationalStrategy(base_variational_strategy, num_tasks=num_classes)
        super().__init__(variational_strategy)
        
        batch_shape = torch.Size([num_classes])
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        
        if kernel == 'rbf':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RBFKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'linear':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.LinearKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'matern':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.MaternKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'polynomial':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PolynomialKernel(power=2, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'piecewise_poly':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PiecewisePolynomialKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'RQ':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RQKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        else:
            raise ValueError(f"Unsupported kernel: {kernel}")

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Load model
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()
for param in model.parameters():
    param.requires_grad = False

# Extract embeddings
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)
train_emb_tensor = torch.tensor(train_emb_pre, dtype=torch.float64).to('cuda')
train_labels_tensor = torch.tensor(train_labels_pre, dtype=torch.long).to('cuda')

print(f"Training set size: {len(train_labels_tensor)}")
print(f"Number of classes: {len(classes)}")
print(f"Samples per class: {len(train_labels_tensor) / len(classes):.1f}")

# Kernel configurations
kernels = [
    ['rbf', 0.01],
    ['matern', 0.01],
    ['linear', 0.001],
    ['polynomial', 0.001],
]

num_inducing = min(100, int(0.5 * len(train_emb_tensor)))
print(f"Using {num_inducing} inducing points")

with gpytorch.settings.cholesky_jitter(1e-4):
    for kernel_name, lr in kernels:
        print(f"\n{'='*60}")
        print(f"Training with {kernel_name} kernel (lr={lr})")
        print(f"{'='*60}")
        
        # Initialize model
        gp_head = MultitaskGPHead(
            num_inducing=num_inducing, 
            feature_dim=768, 
            num_classes=len(classes), 
            kernel=kernel_name
        ).to('cuda').double()
        
        likelihood = gpytorch.likelihoods.SoftmaxLikelihood(
            num_features=len(classes), 
            num_classes=len(classes)
        ).to('cuda').double()
        
        optimizer = torch.optim.Adam([
            {'params': gp_head.parameters()},
            {'params': likelihood.parameters()}
        ], lr=lr)
        
        mll = VariationalELBO(likelihood, gp_head, num_data=len(train_emb_tensor))
        
        # Training
        gp_head.train()
        likelihood.train()
        
        for epoch in range(200):  # Increased epochs
            optimizer.zero_grad()
            output = gp_head(train_emb_tensor)
            loss = -mll(output, train_labels_tensor)
            loss.backward()
            optimizer.step()
            
            if epoch % 20 == 0:
                print(f"Epoch {epoch}: Loss {loss.item():.4f}")
        
        # CORRECTED EVALUATION CODE
        test_emb_pre, test_labels_pre = get_embeddings(model, test_dataloader_manual)
        test_emb_tensor = torch.tensor(test_emb_pre, dtype=torch.float64).to('cuda')  # FIX: float64
        
        gp_head.eval()
        likelihood.eval()
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            # Get GP predictions
            f_pred = gp_head(test_emb_tensor)
            
            # Pass through likelihood to get class probabilities
            observed_pred = likelihood(f_pred)
            
            # Get class predictions - CORRECTED!
            # observed_pred is a Categorical distribution
            pred_probs = observed_pred.probs  # Shape: [num_test, num_classes]
            preds = pred_probs.argmax(dim=-1).cpu().numpy()  # argmax over classes (dim=-1)
        
        acc = accuracy_score(test_labels_pre, preds)
        print(f"\n{kernel_name} kernl - Test Accuracy: {acc:.4f}")
        
        # Debug: Check prediction distribution
        unique, counts = np.unique(preds, return_counts=True)
        print(f"Prediction distribution: {dict(zip(unique, counts))}")
        
        # Clear memory
        del gp_head, likelihood, optimizer
        torch.cuda.empty_cache()


In [ ]:
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, MultitaskVariationalStrategy, VariationalStrategy
from gpytorch.mlls import VariationalELBO
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import ViTForImageClassification

# ==================== GP Model Definition ====================
class MultitaskGPHead(ApproximateGP):
    def __init__(self, num_inducing, feature_dim, num_classes, kernel='rbf'):
        inducing_points = torch.randn(num_inducing, feature_dim, dtype=torch.float64).to('cuda')
        base_variational_distribution = CholeskyVariationalDistribution(num_inducing)
        base_variational_strategy = VariationalStrategy(
            self, inducing_points, base_variational_distribution, learn_inducing_locations=True
        )
        variational_strategy = MultitaskVariationalStrategy(base_variational_strategy, num_tasks=num_classes)
        super().__init__(variational_strategy)
        
        batch_shape = torch.Size([num_classes])
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        
        if kernel == 'rbf':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RBFKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'linear':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.LinearKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'matern':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.MaternKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'polynomial':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PolynomialKernel(power=2, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'piecewise_poly':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PiecewisePolynomialKernel(batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'RQ':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RQKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'periodic':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.PeriodicKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        elif kernel == 'spectral_mixture':
            self.covar_module = gpytorch.kernels.SpectralMixtureKernel(
                num_mixtures=4, ard_num_dims=feature_dim, batch_shape=batch_shape
            )
        else:
            raise ValueError(f"Unsupported kernel: {kernel}")

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


# ==================== Load ViT Model ====================
print("Loading ViT model...")
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()
for param in model.parameters():
    param.requires_grad = False
model.eval()
print("ViT model loaded!\n")


# ==================== Extract Training Embeddings ====================
print("Extracting training embeddings...")
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)

train_emb_tensor = torch.tensor(train_emb_pre, dtype=torch.float64).to('cuda')
train_labels_tensor = torch.tensor(train_labels_pre, dtype=torch.long).to('cuda')

print(f"Training samples: {len(train_labels_tensor)}")
print(f"Embedding dimension: {train_emb_tensor.shape[1]}")
print(f"Number of classes: {len(np.unique(train_labels_pre))}")

classes = np.unique(train_labels_pre)
num_classes = len(classes)


# ==================== Kernel Configurations ====================
kernels = [
    ['rbf', 0.01],
    ['linear', 0.001],
    ['matern', 0.01],
    ['polynomial', 0.001],
    ['piecewise_poly', 0.001],
    ['RQ', 0.01],
    ['periodic', 0.001],
    ['spectral_mixture', 0.001],
]


# ==================== Calculate Inducing Points ====================
num_inducing = max(100, min(500, int(0.5 * len(train_emb_tensor))))
print(f"Using {num_inducing} inducing points out of {len(train_emb_tensor)} training points\n")


# ==================== Training and Evaluation Loop ====================
results = {}

with gpytorch.settings.cholesky_jitter(1e-4):
    for kernel_name, lr in kernels:
        print(f"\n{'='*70}")
        print(f"Training with {kernel_name} kernel (learning rate={lr})")
        print(f"{'='*70}")
        
        try:
            # ==================== Initialize Model ====================
            gp_head = MultitaskGPHead(
                num_inducing=num_inducing, 
                feature_dim=train_emb_tensor.shape[1],
                num_classes=num_classes, 
                kernel=kernel_name
            ).to('cuda').double()
            
            likelihood = gpytorch.likelihoods.SoftmaxLikelihood(
                num_features=num_classes, 
                num_classes=num_classes
            ).to('cuda').double()
            
            optimizer = torch.optim.Adam([
                {'params': gp_head.parameters()},
                {'params': likelihood.parameters()}
            ], lr=lr)
            
            mll = VariationalELBO(likelihood, gp_head, num_data=len(train_emb_tensor))
            
            
            # ==================== Training ====================
            gp_head.train()
            likelihood.train()
            
            num_epochs = 200
            print("Training...")
            for epoch in range(num_epochs):
                optimizer.zero_grad()
                output = gp_head(train_emb_tensor)
                loss = -mll(output, train_labels_tensor)
                loss.backward()
                optimizer.step()
                
                if epoch % 20 == 0:
                    print(f"Epoch {epoch:3d}: Loss {loss.item():.4f}")
            
            print(f"Epoch {num_epochs-1:3d}: Loss {loss.item():.4f}")
            
            
            # ==================== SIMPLE LIST-BASED EVALUATION ====================
            print("\nEvaluating on test set...")
            
            gp_head.eval()
            likelihood.eval()
            
            # Use simple lists (most robust)
            predictions_list = []
            labels_list = []
            
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                for batch_idx, batch in enumerate(test_dataloader_manual):
                    batch_pixels = batch['pixel_values'].to('cuda')
                    batch_labels = batch['label']
                    
                    # Get embeddings
                    outputs = model(pixel_values=batch_pixels)
                    batch_emb = outputs.logits.double()
                    
                    # Get GP predictions
                    f_pred = gp_head(batch_emb)
                    
                    # Transpose: [num_classes, batch_size] -> [batch_size, num_classes]
                    logits = f_pred.mean.transpose(-1, -2)
                    
                    # Apply softmax
                    probs = torch.softmax(logits, dim=-1)
                    
                    # Get predictions
                    batch_preds = probs.argmax(dim=-1)
                    
                    # Debug first batch
                    if batch_idx == 0:
                        print(f"Batch 0 shapes: logits={logits.shape}, probs={probs.shape}, preds={batch_preds.shape}")
                    
                    # Append to lists (handles any batch size)
                    predictions_list.append(batch_preds.cpu())
                    labels_list.append(batch_labels.cpu())
                    
                    del batch_pixels, batch_emb, f_pred, logits, probs
                    torch.cuda.empty_cache()
            
            # Concatenate and convert to numpy
            all_predictions = torch.cat(predictions_list).numpy()
            all_labels = torch.cat(labels_list).numpy()
            
            print(f"Total predictions: {len(all_predictions)}")
            print(f"Total labels: {len(all_labels)}")
            
            # Calculate accuracy
            acc = accuracy_score(all_labels, all_predictions)
            print(f"\n{kernel_name} kernel - Test Accuracy: {acc:.4f}")
            
            results[kernel_name] = acc
            
            # Show prediction distribution
            unique, counts = np.unique(all_predictions, return_counts=True)
            print(f"Prediction distribution: {dict(zip(unique, counts))}")
            
            # Cleanup
            del gp_head, likelihood, optimizer, predictions_list, labels_list
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"\nERROR with {kernel_name} kernel: {str(e)}")
            import traceback
            traceback.print_exc()
            results[kernel_name] = None
            torch.cuda.empty_cache()
            continue


# ==================== Summary ====================
print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
for kernel_name, acc in results.items():
    if acc is not None:
        print(f"{kernel_name:20s}: {acc:.4f}")
    else:
        print(f"{kernel_name:20s}: FAILED")
print("="*70)


In [ ]:
%set_env CUDA_VISIBLE_DEVICES=0

import torch
import tqdm
import gpytorch
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution
from gpytorch.distributions import MultivariateNormal
from gpytorch.models import ApproximateGP, GP
from gpytorch.mlls import VariationalELBO, AddedLossTerm
from gpytorch.likelihoods import GaussianLikelihood

In [ ]:
from gpytorch.models.deep_gps import DeepGPLayer, DeepGP
from gpytorch.mlls import DeepApproximateMLL

In [ ]:
import urllib.request
import os
from scipy.io import loadmat
from math import floor


# this is for running the notebook in our testing framework
smoke_test = ('CI' in os.environ)


if not smoke_test and not os.path.isfile('../elevators.mat'):
    print('Downloading \'elevators\' UCI dataset...')
    urllib.request.urlretrieve('https://drive.google.com/uc?export=download&id=1jhWL3YUHvXIaftia4qeAyDwVxo6j1alk', '../elevators.mat')


if smoke_test:  # this is for running the notebook in our testing framework
    X, y = torch.randn(1000, 3), torch.randn(1000)
else:
    data = torch.Tensor(loadmat('../elevators.mat')['data'])
    X = data[:, :-1]
    X = X - X.min(0)[0]
    X = 2 * (X / X.max(0)[0]) - 1
    y = data[:, -1]


train_n = int(floor(0.8 * len(X)))
train_x = X[:train_n, :].contiguous()
train_y = y[:train_n].contiguous()

test_x = X[train_n:, :].contiguous()
test_y = y[train_n:].contiguous()

if torch.cuda.is_available():
    train_x, train_y, test_x, test_y = train_x.cuda(), train_y.cuda(), test_x.cuda(), test_y.cuda()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(train_x, train_y)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [ ]:
smoke_test = True
num_hidden_dims = 2 if smoke_test else 10


class DeepGP(DeepGP):
    def __init__(self, train_x_shape):
        hidden_layer = ToyDeepGPHiddenLayer(
            input_dims=train_x_shape[-1],
            output_dims=num_hidden_dims,
            mean_type='linear',
        )

        last_layer = ToyDeepGPHiddenLayer(
            input_dims=hidden_layer.output_dims,
            output_dims=None,
            mean_type='constant',
        )

        super().__init__()

        self.hidden_layer = hidden_layer
        self.last_layer = last_layer
        self.likelihood = GaussianLikelihood()

    def forward(self, inputs):
        hidden_rep1 = self.hidden_layer(inputs)
        output = self.last_layer(hidden_rep1)
        return output

    def predict(self, test_loader):
        with torch.no_grad():
            mus = []
            variances = []
            lls = []
            for x_batch, y_batch in test_loader:
                preds = self.likelihood(self(x_batch))
                mus.append(preds.mean)
                variances.append(preds.variance)
                lls.append(model.likelihood.log_marginal(y_batch, model(x_batch)))

        return torch.cat(mus, dim=-1), torch.cat(variances, dim=-1), torch.cat(lls, dim=-1)